# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [3]:
# 1. Load the data using the relative path inside the project directory
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)

# 2. Inspect datatypes and general structure
print("--- Dataframe Info & Datatypes ---")
df.info()

# 3. Inspect NaNs (missing values count per column)
print("\n--- Missing Values (NaNs) Per Column ---")
print(df.isna().sum())

# 4. Print out summary statistics for numerical variables
print("\n--- Summary Statistics ---")
df.describe()

--- Dataframe Info & Datatypes ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Mak

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [4]:
# ==============================================================================
# Data Cleaning: Filtering aircraft and events
# ==============================================================================

# 1. Inspect relevant columns before filtering
print("--- Pre-Filter Distributions ---")
print("Aircraft.Category unique values:\n", df['Aircraft.Category'].value_counts(dropna=False).head())
print("\nAmateur.Built unique values:\n", df['Amateur.Built'].value_counts(dropna=False))

# 2. Extract Event Year safely
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df['Year'] = df['Event.Date'].dt.year

# 3. Standardize strings to uniform uppercase and remove blank spaces to avoid missing matches
df['Aircraft.Category'] = df['Aircraft.Category'].astype(str).str.upper().str.strip()
df['Amateur.Built'] = df['Amateur.Built'].astype(str).str.upper().str.strip()

# 4. Filter the dataset based on strict insurance parameters:
#   - Year >= 1983 (Max 40-year service life rule)
#   - Amateur.Built == 'NO' (Professional builds only)
#   - Aircraft.Category matches 'AIRPLANE' or 'NAN' / 'NONE' 
#     (We preserve missing fields because older NTSB records left this blank for standard planes)
df_filtered = df[
    (df['Year'] >= 1983) & 
    (df['Amateur.Built'] == 'NO') & 
    (df['Aircraft.Category'].isin(['AIRPLANE', 'NAN', 'NONE']))
].copy()

# 5. Output metrics for CodeGrade verification
print("\n--- Post-Filter Results ---")
print(f"Filtered DataFrame Dimensions: {df_filtered.shape}")
print(f"Earliest Event Year: {df_filtered['Year'].min()}")
print(f"Latest Event Year: {df_filtered['Year'].max()}")

--- Pre-Filter Distributions ---
Aircraft.Category unique values:
 Aircraft.Category
NaN           56602
Airplane      27617
Helicopter     3440
Glider          508
Balloon         231
Name: count, dtype: int64

Amateur.Built unique values:
 Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64

--- Post-Filter Results ---
Filtered DataFrame Dimensions: (73002, 32)
Earliest Event Year: 1983
Latest Event Year: 2022


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

### Cleaning Assumptions & Derived Columns Documentation
1. **Imputation Assumption:** Missing (`NaN`) values within the outcome columns (`Total.Fatal.Injuries`, `Total.Serious.Injuries`, `Total.Minor.Injuries`, `Total.Uninjured`) are filled with `0`. Historically, if an outcome category is left empty in NTSB reports, it indicates there were zero passengers matching that state.
2. **Total Aboard Feature:** Calculated by summing all four outcome categories. This establishes an accurate flight capacity denominator per incident.
3. **Severe Injury Rate Feature:** An actuarial risk metric calculated by dividing the sum of fatal and serious injuries by the total number of people aboard. A condition handling `Total_Aboard == 0` prevents mathematical zero-division errors.

In [5]:
# ==============================================================================
# Cleaning and Constructing Key Measurables
# ==============================================================================

# List of columns representing all possible individual outcomes on a flight
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Step 1: Impute missing injury records with 0 to enable proper arithmetic operations
for col in injury_cols:
    df_filtered[col] = df_filtered[col].fillna(0)

# Step 2: Derive 'Total_Aboard' to estimate total passenger footprint per flight
df_filtered['Total_Aboard'] = df_filtered[injury_cols].sum(axis=1)

# Step 3: Compute total severe outcomes (Fatalities + Serious Injuries)
df_filtered['Severe_Injury_Count'] = df_filtered['Total.Fatal.Injuries'] + df_filtered['Total.Serious.Injuries']

# Step 4: Construct the risk metric fraction (Severe Injury Rate)
# np.where evaluates if passengers were present to avoid 0-division runtime errors
df_filtered['Severe_Injury_Rate'] = np.where(
    df_filtered['Total_Aboard'] > 0,
    df_filtered['Severe_Injury_Count'] / df_filtered['Total_Aboard'],
    0.0
)

# Step 5: Verify the engineered columns
print("--- Summary Statistics for New Human Impact Measurables ---")
print(df_filtered[['Total_Aboard', 'Severe_Injury_Count', 'Severe_Injury_Rate']].describe())

--- Summary Statistics for New Human Impact Measurables ---
       Total_Aboard  Severe_Injury_Count  Severe_Injury_Rate
count  73002.000000         73002.000000        73002.000000
mean       6.914400             0.830964            0.266812
std       30.321715             5.935758            0.425764
min        0.000000             0.000000            0.000000
25%        1.000000             0.000000            0.000000
50%        2.000000             0.000000            0.000000
75%        3.000000             1.000000            0.500000
max      699.000000           349.000000            1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

### Aircraft Damage Cleaning & Feature Engineering Documentation
1. **Cleaning and Imputation Assumption:** The raw `Aircraft.damage` column contains mixed case text strings and a small number of missing values (`NaN`). Missing values are filled with `'UNKNOWN'` to prevent string function errors while keeping rows intact. All valid strings are converted to uniform uppercase and stripped of whitespace to eliminate duplicates.
2. **Derived Binary Target (`Is_Destroyed`):** To provide a clean performance metric for the insurer, a derived boolean column `Is_Destroyed` is created. It maps occurrences of `'TOTAL DESTROYED'` to `1` and all lower damage severities (`SUBSTANTIAL`, `MINOR`, `UNKNOWN`) to `0`. This gives us a mathematically reliable way to calculate hull destruction rates per aircraft manufacturer.

In [7]:
# ==============================================================================
# Data Cleaning & Feature Engineering: Aircraft Damage
# ==============================================================================

# 1. Fill missing values with a placeholder to allow safe string logic
df_filtered['Aircraft.damage'] = df_filtered['Aircraft.damage'].fillna('UNKNOWN')

# 2. Convert values to uniform uppercase and strip trailing/leading spaces
df_filtered['Aircraft.damage'] = df_filtered['Aircraft.damage'].astype(str).str.upper().str.strip()

# 3. Create the derived binary column tracking complete hull destruction
# (1 = Total Destroyed, 0 = Survived with Minor/Substantial damage or Unknown status)
df_filtered['Is_Destroyed'] = (df_filtered['Aircraft.damage'] == 'TOTAL DESTROYED').astype(int)

# 4. Print validation summaries for the project report
print("--- Cleaned Aircraft Damage Text Categories ---")
print(df_filtered['Aircraft.damage'].value_counts())

print("\n--- Derived Flag Distribution (Is_Destroyed) ---")
print(df_filtered['Is_Destroyed'].value_counts(normalize=True))

--- Cleaned Aircraft Damage Text Categories ---
Aircraft.damage
SUBSTANTIAL    52547
DESTROYED      14958
UNKNOWN         3023
MINOR           2474
Name: count, dtype: int64

--- Derived Flag Distribution (Is_Destroyed) ---
Is_Destroyed
0    1.0
Name: proportion, dtype: float64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

### Model Column Diagnostics & Unique Type Formulation

1. **Imputation & Missing Value Removal:** Any missing (`NaN`) values within the `Model` column are filled with `'UNKNOWN'` to maintain string manipulation stability and ensure no data records are dropped during string grouping operations.
2. **Text Standardization:** Model strings are converted to uniform uppercase and stripped of any extraneous leading or trailing white spaces to resolve variations caused by inconsistent entry formats.
3. **Cross-Manufacturer Duplicate Collision (Non-Uniqueness):** Model designations are **not** unique to individual makes. For example, both Cessna and Piper might utilize standard numeric designations (like '150' or '172') or alpha designations, which would cause inaccurate aggregated findings if grouped solely by model.
4. **Derived Unique Identifier Type (`Aircraft_Type`):** To resolve this non-uniqueness issue, a derived column named `Aircraft_Type` is created by concatenating the cleaned `Make` and `Model` strings with a delimiter (e.g., `"CESSNA | 172"`). This guarantees a globally unique identifier for every specific aircraft type in the dataset, fulfilling the insurance client's requirement to track distinct models accurately.

In [8]:
# ==============================================================================
# Data Cleaning & Thresholding: Aircraft Make
# ==============================================================================

# Step 1: Handle missing values safely
df_filtered['Make'] = df_filtered['Make'].fillna('UNKNOWN')

# Step 2: Standardize case and strip accidental trailing or leading spaces
df_filtered['Make'] = df_filtered['Make'].astype(str).str.upper().str.strip()

# Step 3: Consolidate structural naming variations for major corporate brands
manufacturer_mapping = {
    'CESSNA AIRCRAFT CO': 'CESSNA',
    'CESSNA AIRCRAFT CO.': 'CESSNA',
    'CESSNA AIRCRAFT': 'CESSNA',
    'PIPER AIRCRAFT CORP': 'PIPER',
    'PIPER AIRCRAFT CORPORATION': 'PIPER',
    'BEECH AIRCRAFT CORP': 'BEECH',
    'BEECH AIRCRAFT CORPORATION': 'BEECH',
    'BOEING COMPANY': 'BOEING',
    'BOEING-VERTOL': 'BOEING',
    'AIRBUS INDUSTRIE': 'AIRBUS',
    'MCDONNELL DOUGLAS AIRCRAFT CO': 'MCDONNELL DOUGLAS',
    'MCDONNELL DOUGLAS CORP': 'MCDONNELL DOUGLAS'
}
df_filtered['Make'] = df_filtered['Make'].replace(manufacturer_mapping)

# Step 4: Identify and keep Makes meeting the statistical threshold (>= 50 incidents)
make_counts = df_filtered['Make'].value_counts()
robust_makes = make_counts[make_counts >= 50].index

# Filter the main dataframe to only keep these statistically robust makes
df_robust_makes = df_filtered[df_filtered['Make'].isin(robust_makes)].copy()

# Step 5: Print verification data for your report
print(f"Total unique Makes before cleanup: {len(make_counts)}")
print(f"Total unique Makes after thresholding (>= 50 incidents): {len(robust_makes)}")
print("\n--- Top 10 Manufacturers by Incident Volume ---")
print(df_robust_makes['Make'].value_counts().head(10))

Total unique Makes before cleanup: 1543
Total unique Makes after thresholding (>= 50 incidents): 85

--- Top 10 Manufacturers by Incident Volume ---
Make
CESSNA      25775
PIPER       14110
BEECH        5099
BOEING       2655
BELL         1793
MOONEY       1271
GRUMMAN      1064
BELLANCA      978
HUGHES        686
ROBINSON      676
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [9]:
# ==============================================================================
# Data Cleaning & Feature Engineering: Aircraft Model
# ==============================================================================

# Step 1: Fill NaNs and standardize the text case/spacing
df_robust_makes['Model'] = df_robust_makes['Model'].fillna('UNKNOWN')
df_robust_makes['Model'] = df_robust_makes['Model'].astype(str).str.upper().str.strip()

# Step 2: Diagnostic Check - Inspect whether model names are unique across makes
# Let's count how many distinct manufacturers use the exact same model name
model_uniqueness = df_robust_makes.groupby('Model')['Make'].nunique()
shared_models = model_uniqueness[model_uniqueness > 1]

print("--- Cross-Manufacturer Model Uniqueness Check ---")
print(f"Number of model names shared by multiple manufacturers: {len(shared_models)}")
if len(shared_models) > 0:
    print("\nExamples of shared model labels across different makes:")
    # Look up examples of a common duplicated model designation
    example_model = shared_models.index[0]
    print(df_robust_makes[df_robust_makes['Model'] == example_model][['Make', 'Model']].drop_duplicates().head(3))
else:
    print("All model labels are uniquely mapped to individual manufacturers.")

# Step 3: Construct a globally unique derived identifier combining Make and Model
df_robust_makes['Aircraft_Type'] = df_robust_makes['Make'] + " | " + df_robust_makes['Model']

# Step 4: Verify the newly generated unique aircraft type column
print("\n--- Unique Aircraft Type Feature Verification ---")
print(f"Total unique aircraft types engineered: {df_robust_makes['Aircraft_Type'].nunique()}")
print("\nTop 5 unique aircraft configurations by incident volume:")
print(df_robust_makes['Aircraft_Type'].value_counts().head(5))

--- Cross-Manufacturer Model Uniqueness Check ---
Number of model names shared by multiple manufacturers: 418

Examples of shared model labels across different makes:
                 Make Model
6339   AERO COMMANDER   100
14526        ROCKWELL   100
16477           BEECH   100

--- Unique Aircraft Type Feature Verification ---
Total unique aircraft types engineered: 6221

Top 5 unique aircraft configurations by incident volume:
Aircraft_Type
CESSNA | 152         2229
CESSNA | 172         1650
CESSNA | 172N        1093
PIPER | PA-28-140     863
CESSNA | 172M         759
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [10]:
# ==============================================================================
# Contextual Variables Cleaning & Standardization
# ==============================================================================

context_cols = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']

print("--- Pre-Cleaning Structural Value Snapshots ---")
for col in context_cols:
    print(f"\nUnique values for {col} (First 3 rows):")
    print(df_robust_makes[col].value_counts(dropna=False).head(3))

# Step 1: Execute string normalization across all object columns simultaneously
for col in context_cols:
    # Fill structural missing tags safely without dropping rows
    df_robust_makes[col] = df_robust_makes[col].fillna('UNKNOWN')
    # Force uppercase and remove accidental trail/lead string spaces
    df_robust_makes[col] = df_robust_makes[col].astype(str).str.upper().str.strip()

# Step 2: Clean up explicit string anomalies within Weather.Condition
weather_mapping = {
    'VMC': 'VMC',
    'IMC': 'IMC',
    'UNK': 'UNKNOWN',
    'NONE': 'UNKNOWN'
}
df_robust_makes['Weather.Condition'] = df_robust_makes['Weather.Condition'].replace(weather_mapping)

# Step 3: Handle Number.of.Engines column values cleanly
df_robust_makes['Number.of.Engines'] = df_robust_makes['Number.of.Engines'].fillna(-1.0) # Using -1 to indicate unknown numerical status cleanly

# Step 4: Verify the streamlined categorical outputs
print("\n" + "="*80)
print("--- Post-Cleaning Verification Results ---")
for col in context_cols + ['Number.of.Engines']:
    print(f"\nCleaned Category Distribution for: {col}")
    print(df_robust_makes[col].value_counts().head(3))

--- Pre-Cleaning Structural Value Snapshots ---

Unique values for Engine.Type (First 3 rows):
Engine.Type
Reciprocating    53960
NaN               4597
Turbo Prop        2796
Name: count, dtype: int64

Unique values for Weather.Condition (First 3 rows):
Weather.Condition
VMC    58313
IMC     5177
NaN     3237
Name: count, dtype: int64

Unique values for Purpose.of.flight (First 3 rows):
Purpose.of.flight
Personal         36268
Instructional     8881
Unknown           5400
Name: count, dtype: int64

Unique values for Broad.phase.of.flight (First 3 rows):
Broad.phase.of.flight
NaN        18067
Landing    12874
Takeoff     9740
Name: count, dtype: int64

--- Post-Cleaning Verification Results ---

Cleaned Category Distribution for: Engine.Type
Engine.Type
RECIPROCATING    53960
UNKNOWN           6052
TURBO PROP        2796
Name: count, dtype: int64

Cleaned Category Distribution for: Weather.Condition
Weather.Condition
VMC        58313
IMC         5177
UNKNOWN     4133
Name: count, dtype

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [11]:
# ==============================================================================
# Missing Data Density Audit & Column Drop Execution
# ==============================================================================

# Step 1: Calculate the percentage of missing values (NaNs) for each column
missing_percentages = df_robust_makes.isnull().sum() / len(df_robust_makes)

print("--- Missing Data Profile (Top 10 Empty Columns) ---")
print(missing_percentages.sort_values(ascending=False).head(10))

# Step 2: Set our dropping threshold criteria (Greater than 50% missing)
drop_threshold = 0.50
columns_to_drop = missing_percentages[missing_percentages > drop_threshold].index.tolist()

print(f"\nColumns flagged for removal (exceeding {drop_threshold*100}% missing data):")
print(columns_to_drop)

# Step 3: Execute column removal on the working dataframe
df_clean_final = df_robust_makes.drop(columns=columns_to_drop).copy()

# Step 4: Verify the final dataset dimension footprint
print("\n--- Dataframe Shape Comparison ---")
print(f"Original working shape: {df_robust_makes.shape}")
print(f"Cleaned working shape:  {df_clean_final.shape}")

--- Missing Data Profile (Top 10 Empty Columns) ---
Schedule               0.844032
Air.carrier            0.827529
FAR.Description        0.724073
Longitude              0.652574
Latitude               0.652515
Airport.Code           0.426467
Airport.Name           0.399184
Publication.Date       0.178061
Report.Status          0.064342
Registration.Number    0.015512
dtype: float64

Columns flagged for removal (exceeding 50.0% missing data):
['Latitude', 'Longitude', 'FAR.Description', 'Schedule', 'Air.carrier']

--- Dataframe Shape Comparison ---
Original working shape: (67623, 37)
Cleaned working shape:  (67623, 32)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [12]:
# ==============================================================================
# Saving Cleaned Data Checkpoint to CSV
# ==============================================================================
import os

# Define the file path for the clean dataset
output_filename = "aviation_accidents_cleaned.csv"

# Export the final processed DataFrame to a CSV file
# Index=False prevents Pandas from adding an empty, unnamed row index column
df_clean_final.to_csv(output_filename, index=False)

print("--- Workflow Checkpoint Successfully Saved ---")
print(f"File saved as: {os.path.abspath(output_filename)}")
print(f"Final dataset dimensions for EDA: {df_clean_final.shape}")

--- Workflow Checkpoint Successfully Saved ---
File saved as: C:\Users\BRIDGIT\dsc-course0-m8-lab\aviation_accidents_cleaned.csv
Final dataset dimensions for EDA: (67623, 32)
